In [ ]:
# 필요한 라이브러리가 잘 불러와지는지 확인
import sys
import json
from pathlib import Path

print(sys.executable)  # .venv 안의 파이썬인지 확인용

try:
    import pandas as pd
    print("pandas 있음:", pd.__version__)
except ImportError:
    print("pandas 없음")

import matplotlib
print("matplotlib 있음:", matplotlib.__version__)

# 데이터 구조 확인

훈련 전에 정해야 할 것들(라벨 변환 방법, train/val 분할 기준, 외부 데이터 필요 여부)을 데이터 보면서 정리해본다.

## 1. 파일 개수

In [ ]:
# 위로 올라가면서 pyproject.toml 있는 폴더를 레포 루트로 잡음
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "raw" / "sprint_ai_project1_data_20261022" / "sprint_ai_project1_data"

TRAIN_IMG_DIR = DATA_DIR / "train_images"
TRAIN_ANN_DIR = DATA_DIR / "train_annotations"
TEST_IMG_DIR = DATA_DIR / "test_images"

# 파일 목록 모으기 (rglob은 하위 폴더까지 다 뒤짐)
train_imgs = sorted(TRAIN_IMG_DIR.glob("*.png"))
test_imgs = sorted(TEST_IMG_DIR.glob("*.png"))
json_files = sorted(TRAIN_ANN_DIR.rglob("*.json"))

print("ROOT:", ROOT)
print("train 이미지:", len(train_imgs))
print("test 이미지:", len(test_imgs))
print("json 파일:", len(json_files))

## 2. json 읽어서 표로 만들기

json 하나에 알약 하나라서, 한 줄 = 알약 하나로 정리한다.

In [ ]:
rows = []
for jp in json_files:
    with open(jp, encoding="utf-8") as f:
        d = json.load(f)

    img = d["images"][0]  # 이미지 정보
    for ann in d["annotations"]:  # 보통 1개인데 혹시 몰라서 반복
        x, y, w, h = ann["bbox"]
        rows.append({
            "file_name": img["file_name"],
            "img_w": img["width"],
            "img_h": img["height"],
            "category_id": ann["category_id"],
            "drug_name": img["dl_name"],
            "x": x, "y": y, "w": w, "h": h,
            "camera_la": img["camera_la"],
            "back_color": img["back_color"],
            "json_path": str(jp.relative_to(TRAIN_ANN_DIR)),
        })

ann_df = pd.DataFrame(rows)
print(ann_df.shape)  # (줄 수, 열 수)
ann_df.head()

## 3. 라벨 빠진 이미지 확인

파일 이름 앞부분(K-xxxxxx-xxxxxx-...)에 사진 속 약 코드가 들어 있다.
이름 속 약 개수랑 실제 json 개수가 다르면 라벨이 빠진 것.

In [ ]:
# 이미지 한 장 = 한 줄인 표
img_df = pd.DataFrame({"file_name": [p.name for p in train_imgs]})

# 이름 앞부분에서 약 개수 세기 (K-001900-016548 → '-' 2개 = 약 2개)
img_df["combo"] = img_df["file_name"].str.split("_").str[0]
img_df["n_expected"] = img_df["combo"].str.count("-")

# 실제 라벨 개수 (json이 하나도 없는 이미지는 0)
n_ann = ann_df.groupby("file_name").size()
img_df["n_ann"] = img_df["file_name"].map(n_ann).fillna(0).astype(int)

img_df["n_missing"] = img_df["n_expected"] - img_df["n_ann"]

print("이름 속 약 개수별 이미지 수")
print(img_df["n_expected"].value_counts().sort_index())
print()
print("빠진 라벨 개수별 이미지 수")
print(img_df["n_missing"].value_counts().sort_index())

알약 3개짜리 157장, 4개짜리 75장. 라벨이 하나씩 빠진 이미지가 8장 있다.
어떤 약이 빠졌는지 확인해본다.

In [ ]:
pd.set_option("display.max_colwidth", None)

missing = img_df[img_df["n_missing"] > 0].copy()

def find_missing_ids(row):
    # 이름 속 약 코드 (K-001900 → 1900)
    expected = {int(c) for c in row["combo"].split("-")[1:]}
    # 실제 라벨에 있는 category_id
    got = set(ann_df.loc[ann_df["file_name"] == row["file_name"], "category_id"])
    return sorted(expected - got)

missing["missing_ids"] = missing.apply(find_missing_ids, axis=1)
missing[["file_name", "n_expected", "n_ann", "missing_ids"]]

빠진 8장 중 7장에 K-003351이 들어 있다. 특정 약 묶음에서 라벨 작업이 빠진 걸로 보임.
폴더 자체가 없는 건지, 폴더는 있는데 이 각도 json만 없는 건지 확인.

In [ ]:
for _, row in missing.iterrows():
    combo_dir = TRAIN_ANN_DIR / f"{row['combo']}_json"  # 이 알약 조합의 json 폴더

    # 약 종류별 하위 폴더 목록
    sub_dirs = sorted(p.name for p in combo_dir.iterdir()) if combo_dir.exists() else ["조합 폴더 없음"]

    # 같은 조합의 다른 각도 사진들
    same_combo = img_df.loc[img_df["combo"] == row["combo"], "file_name"].tolist()

    print(row["file_name"])
    print("  빠진 약:", row["missing_ids"])
    print("  있는 약 폴더:", sub_dirs)
    print("  같은 조합 사진 수:", len(same_combo))
    print()

빠진 약 폴더는 다 있다. 다른 각도 json은 있는데 이 각도 json만 없는 것.
각도마다 위치가 달라서 복사로는 못 채우고 직접 박스를 그려야 함.

## 4. 라벨 일관성 점검

In [ ]:
img_names = set(img_df["file_name"])

# 1) json에 적힌 이미지가 실제로 없는 경우
orphan = ann_df[~ann_df["file_name"].isin(img_names)]
print("이미지 없는 라벨:", len(orphan))

# 2) json 파일 이름이랑 안에 적힌 file_name이 다른 경우
json_img_name = ann_df["json_path"].apply(lambda p: Path(p).stem + ".png")
print("json 이름과 file_name 불일치:", (json_img_name != ann_df["file_name"]).sum())

# 3) 같은 이미지에 같은 약 라벨이 2개 이상
dup = ann_df[ann_df.duplicated(["file_name", "category_id"], keep=False)]
print("같은 이미지에 같은 약 중복:", len(dup))

# 4) 약 폴더 이름(K-001900)이랑 category_id(1900)가 다른 경우
folder_id = ann_df["json_path"].apply(lambda p: int(Path(p).parent.name.split("-")[1]))
print("폴더 이름과 category_id 불일치:", (folder_id != ann_df["category_id"]).sum())

라벨-이미지 연결은 다 정상. 문제는 빠진 8장뿐이 아님

## 5. 박스 좌표 점검 (8장이 다가 아니란거 앎. 다시 확인 시작)

In [ ]:
from PIL import Image

# 박스 오른쪽 아래 좌표
ann_df["x2"] = ann_df["x"] + ann_df["w"]
ann_df["y2"] = ann_df["y"] + ann_df["h"]

# 1) 크기가 0 이하인 박스
print("크기 0 이하:", ((ann_df["w"] <= 0) | (ann_df["h"] <= 0)).sum())

# 2) 이미지 밖으로 나간 박스
out = (ann_df["x"] < 0) | (ann_df["y"] < 0) | (ann_df["x2"] > ann_df["img_w"]) | (ann_df["y2"] > ann_df["img_h"])
print("이미지 밖으로 나간 박스:", out.sum())

# 3) 실제 이미지 크기 읽기 (헤더만 읽어서 빠름)
def get_size(p):
    with Image.open(p) as im:
        return im.size  # (가로, 세로)

img_df["real_w"], img_df["real_h"] = zip(*[get_size(TRAIN_IMG_DIR / n) for n in img_df["file_name"]])

# json에 적힌 크기랑 실제 크기 비교
merged = ann_df.merge(img_df[["file_name", "real_w", "real_h"]], on="file_name")
size_diff = (merged["img_w"] != merged["real_w"]) | (merged["img_h"] != merged["real_h"])
print("json 크기와 실제 크기 다른 라벨:", size_diff.sum())

# 실제 이미지 해상도 종류
print()
print(img_df.groupby(["real_w", "real_h"]).size())

In [ ]:
# 약별 박스 넓이의 중앙값과 비교
ann_df["area"] = ann_df["w"] * ann_df["h"]
ann_df["area_ratio"] = ann_df["area"] / ann_df.groupby("category_id")["area"].transform("median")

# 보통 크기의 절반 이하거나 2배 이상이면 의심
size_odd = ann_df[(ann_df["area_ratio"] < 0.5) | (ann_df["area_ratio"] > 2)]
print("크기 이상 의심 박스:", len(size_odd), "/ 이미지 수:", size_odd["file_name"].nunique())
size_odd[["file_name", "drug_name", "w", "h", "area_ratio"]].sort_values("area_ratio")

## 6. 전체 이미지 눈으로 확인

파일명 기준 검사는 이름 자체가 틀리면 못 잡는다. 박스 그려서 전부 직접 본다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import rc

rc("font", family="Malgun Gothic")  # 윈도우 한글 폰트
plt.rcParams["axes.unicode_minus"] = False

def show_page(page, per_page=12):
    names = img_df["file_name"].tolist()[page * per_page:(page + 1) * per_page]
    fig, axes = plt.subplots(3, 4, figsize=(16, 16))

    for ax, name in zip(axes.flat, names):
        ax.imshow(Image.open(TRAIN_IMG_DIR / name))
        anns = ann_df[ann_df["file_name"] == name]
        for _, a in anns.iterrows():
            ax.add_patch(patches.Rectangle((a["x"], a["y"]), a["w"], a["h"],
                                           fill=False, edgecolor="lime", linewidth=2))
        idx = img_df.index[img_df["file_name"] == name][0]
        ax.set_title(f"[{idx}] 이름 {img_df.loc[idx, 'n_expected']}개 / 박스 {len(anns)}개", fontsize=10)
        ax.axis("off")

    for ax in axes.flat[len(names):]:  # 남는 칸은 비우기
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_page(0)  # 0번 페이지 (이미지 0~11번)

In [ ]:
# [3]번 이미지 박스 좌표 보기
name3 = img_df.loc[3, "file_name"]
print(ann_df.loc[ann_df["file_name"] == name3, ["drug_name", "category_id", "x", "y", "w", "h", "x2", "y2"]])
print()

# 두 박스가 얼마나 겹치는지 (IoU: 0이면 안 겹침, 1이면 완전히 같은 위치)
def iou(a, b):
    ix = max(0, min(a["x2"], b["x2"]) - max(a["x"], b["x"]))
    iy = max(0, min(a["y2"], b["y2"]) - max(a["y"], b["y"]))
    inter = ix * iy
    return inter / (a["w"] * a["h"] + b["w"] * b["h"] - inter)

# 같은 이미지 안에서 많이 겹치는 박스 쌍 찾기
overlaps = []
for name, g in ann_df.groupby("file_name"):
    rows = g.to_dict("records")
    for i in range(len(rows)):
        for j in range(i + 1, len(rows)):
            v = iou(rows[i], rows[j])
            if v > 0.3:  # 알약끼리는 거의 안 겹치니까 0.3만 넘어도 의심
                overlaps.append((name, rows[i]["drug_name"], rows[j]["drug_name"], round(v, 2)))

print("많이 겹치는 박스 쌍:", len(overlaps))
pd.DataFrame(overlaps, columns=["file_name", "약1", "약2", "IoU"])

서로 다른 약에 똑같은 박스가 붙은 이미지 3장 발견 (복붙 실수로 보임).
2장은 또 K-003351 묶음.

## 7. 문제 이미지 목록

In [ ]:
issues = []

# 1) 라벨 누락
for name in img_df.loc[img_df["n_missing"] > 0, "file_name"]:
    issues.append((name, "라벨 누락"))

# 2) 이미지 밖으로 나간 박스
for name in ann_df.loc[out, "file_name"]:
    issues.append((name, "박스 이미지 밖"))

# 3) 다른 약에 같은 박스 복사
for name, *_ in overlaps:
    issues.append((name, "박스 복사"))

issue_df = pd.DataFrame(issues, columns=["file_name", "문제"])

print("문제 건수:", len(issue_df))
print("문제 이미지 수 (중복 제외):", issue_df["file_name"].nunique())
print()
print(issue_df["문제"].value_counts())

# 한 이미지에 문제가 여러 개인 경우
multi = issue_df.groupby("file_name")["문제"].apply(list)
multi[multi.str.len() > 1]

지금까지 12장. 박스 크기·비율이 그 약의 평소 모양과 많이 다른 것도 찾아본다.

In [ ]:
# 약별 평소 크기, 비율(중앙값)과 비교
ann_df["area"] = ann_df["w"] * ann_df["h"]
ann_df["ratio"] = ann_df["w"] / ann_df["h"]

g = ann_df.groupby("category_id")
ann_df["area_vs_med"] = ann_df["area"] / g["area"].transform("median")

# 비율은 각도에 따라 가로세로가 뒤집힐 수 있어서 긴 변/짧은 변으로 비교
ann_df["long_short"] = ann_df[["w", "h"]].max(axis=1) / ann_df[["w", "h"]].min(axis=1)
ann_df["ls_vs_med"] = ann_df["long_short"] / g["long_short"].transform("median")

# 크기가 절반 이하/2배 이상, 또는 비율이 1.3배 넘게 차이 나면 후보
cand = ann_df[
    (ann_df["area_vs_med"] < 0.5) | (ann_df["area_vs_med"] > 2) |
    (ann_df["ls_vs_med"] < 1 / 1.3) | (ann_df["ls_vs_med"] > 1.3)
]

# 이미 찾은 12장은 빼고 보기
cand = cand[~cand["file_name"].isin(issue_df["file_name"])]

print("새 후보 박스:", len(cand), "/ 이미지:", cand["file_name"].nunique())
cand[["file_name", "drug_name", "w", "h", "area_vs_med", "ls_vs_med"]].round(2)

In [ ]:
# 평소 모양에서 얼마나 벗어났는지 점수 (1에서 멀수록 이상)
import numpy as np
cand = cand.copy()
cand["score"] = (np.log(cand["area_vs_med"]).abs() + np.log(cand["ls_vs_med"]).abs())

print("후보 박스:", len(cand), "/ 이미지:", cand["file_name"].nunique())
cand.sort_values("score", ascending=False)[["file_name", "drug_name", "w", "h", "area_vs_med", "ls_vs_med"]].head(10).round(2)

크기는 다 정상 범위. 비율 차이는 알약이 기울어져 있어서 생긴 것.
문제가 K-003351 묶음에 몰려 있으니 거기부터 눈으로 확인.

In [ ]:
def show_images(names, page=0, per_page=12):
    names = names[page * per_page:(page + 1) * per_page]
    fig, axes = plt.subplots(3, 4, figsize=(16, 16))

    for ax, name in zip(axes.flat, names):
        ax.imshow(Image.open(TRAIN_IMG_DIR / name))
        anns = ann_df[ann_df["file_name"] == name]
        for _, a in anns.iterrows():
            ax.add_patch(patches.Rectangle((a["x"], a["y"]), a["w"], a["h"],
                                           fill=False, edgecolor="lime", linewidth=2))
        idx = img_df.index[img_df["file_name"] == name][0]
        ax.set_title(f"[{idx}] 이름 {img_df.loc[idx, 'n_expected']}개 / 박스 {len(anns)}개", fontsize=10)
        ax.axis("off")

    for ax in axes.flat[len(names):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

# K-003351 들어간 이미지 중 아직 문제 목록에 없는 것
names_3351 = img_df.loc[
    img_df["combo"].str.contains("003351") & ~img_df["file_name"].isin(issue_df["file_name"]),
    "file_name"
].tolist()
print("확인할 이미지:", len(names_3351))

show_images(names_3351, page=0)

K-003351 들어간 사진이 147장이라 묶음별로 보는 건 의미가 적음.
박스 안이 실제로 알약으로 차 있는지 자동으로 확인해본다.

In [ ]:
def box_fill(name, boxes):
    im = np.asarray(Image.open(TRAIN_IMG_DIR / name).convert("RGB")).astype(float)
    bg = np.median(im.reshape(-1, 3), axis=0)   # 배경색 (알약이 작아서 중앙값이 거의 배경)
    diff = np.linalg.norm(im - bg, axis=2)      # 픽셀마다 배경과의 색 차이
    mask = diff > 40                            # 차이 크면 알약으로 봄

    res = []
    for idx, a in boxes.iterrows():
        x, y = max(0, int(a["x"])), max(0, int(a["y"]))
        x2, y2 = min(im.shape[1], int(a["x2"])), min(im.shape[0], int(a["y2"]))
        part = mask[y:y2, x:x2]
        res.append((idx, part.mean() if part.size else 0.0))
    return res

fills = []
for name, boxes in ann_df.groupby("file_name"):
    fills += box_fill(name, boxes)

ann_df["fill"] = pd.Series(dict(fills))

print(ann_df["fill"].describe().round(2))
ann_df.sort_values("fill")[["file_name", "drug_name", "w", "h", "fill"]].head(10).round(2)

In [ ]:
# fill 0인 박스가 있는 이미지 + 박스 이미지 밖 이미지
zero_names = ann_df.loc[ann_df["fill"] < 0.05, "file_name"].tolist()
out_names = ann_df.loc[out, "file_name"].tolist()

print("fill 0 이미지:", zero_names)
print("박스 밖 이미지:", out_names)
print("이미 문제 목록에 있는 것:", [n for n in zero_names if n in set(issue_df["file_name"])])

check_names = list(dict.fromkeys(zero_names + out_names))  # 중복 제거, 순서 유지
show_images(check_names)

[71] 캡슐 박스가 이미지 밖에 있음 (살짝 넘친 게 아니라 위치 자체가 틀림)
[123] 박스가 알약 옆 빈 곳에 그려짐
둘 다 다시 그려야 함.

In [ ]:
# [123] 박스 빈 곳 추가
issues.append((img_df.loc[123, "file_name"], "박스 빈 곳"))

issue_df = pd.DataFrame(issues, columns=["file_name", "문제"])
print("문제 이미지 수:", issue_df["file_name"].nunique())
print()
print(issue_df["문제"].value_counts())

# K-003351이 무슨 약인지 확인
print()
print(ann_df.loc[ann_df["category_id"] == 3351, "drug_name"].unique())

K-003351 = 일양하이트린정 2mg. 문제 대부분이 이 약 주변에서 나온다.
이 약 박스만 따로 점검.

In [ ]:
# 문제 이미지 중 일양하이트린정이 들어간 비율
issue_imgs = issue_df["file_name"].unique()
with_3351 = [n for n in issue_imgs if "003351" in n]
print(f"문제 이미지 {len(issue_imgs)}장 중 일양하이트린정 포함: {len(with_3351)}장")

# 이 약 박스의 fill 분포 (다른 약이랑 비교)
is_3351 = ann_df["category_id"] == 3351
print()
print("일양하이트린정 fill:")
print(ann_df.loc[is_3351, "fill"].describe().round(2))
print()
print("나머지 약 fill:")
print(ann_df.loc[~is_3351, "fill"].describe().round(2))

# 이 약 박스 중 fill 낮은 것 (문제 목록에 없는 것만)
low = ann_df[is_3351 & (ann_df["fill"] < 0.5) & ~ann_df["file_name"].isin(issue_imgs)]
print()
print("fill 0.5 미만 일양하이트린정 박스 (새 후보):", len(low))
low[["file_name", "w", "h", "fill"]].round(2)

문제 13장 중 11장이 일양하이트린정 포함. 그런데 이 약 박스 자체는 정확함.
빠뜨리거나 엉뚱한 데 복사한 식의 실수.

## 8. 박스 없는 알약 찾기

파일명에도 없고 박스도 없는 알약은 이름 기준 검사로 못 잡는다.
박스 밖에 남은 큰 덩어리를 찾아본다.

In [ ]:
from scipy import ndimage

def unboxed_blobs(name, boxes, margin=10, min_area=3000):
    im = np.asarray(Image.open(TRAIN_IMG_DIR / name).convert("RGB")).astype(float)
    bg = np.median(im.reshape(-1, 3), axis=0)
    mask = np.linalg.norm(im - bg, axis=2) > 40   # 알약 픽셀

    # 박스 안(+여유 margin)은 지우기
    for _, a in boxes.iterrows():
        x, y = max(0, int(a["x"]) - margin), max(0, int(a["y"]) - margin)
        x2, y2 = int(a["x2"]) + margin, int(a["y2"]) + margin
        mask[y:y2, x:x2] = False

    # 남은 픽셀을 덩어리로 묶기
    labels, n = ndimage.label(mask)
    sizes = np.bincount(labels.ravel())          # 덩어리별 픽셀 수 한 번에 계산
    slices = ndimage.find_objects(labels)        # 덩어리별 위치 범위
    h, w = mask.shape

    found = []
    for i, sl in enumerate(slices, start=1):
        if sizes[i] < min_area:  # 작은 건 잡음
            continue
        ys, xs = sl
        if ys.start == 0 or xs.start == 0 or ys.stop == h or xs.stop == w:
            continue  # 테두리에 닿으면 귀퉁이 조명으로 보고 제외
        found.append(int(sizes[i]))
    return found

rows = []
for name, boxes in ann_df.groupby("file_name"):
    blobs = unboxed_blobs(name, boxes)
    if blobs:
        rows.append((name, len(blobs), max(blobs)))

blob_df = pd.DataFrame(rows, columns=["file_name", "덩어리 수", "제일 큰 덩어리(px)"])
blob_df["이미 찾음"] = blob_df["file_name"].isin(issue_imgs)

print("박스 밖 큰 덩어리 있는 이미지:", len(blob_df))
blob_df.sort_values("제일 큰 덩어리(px)", ascending=False)

In [ ]:
# 1) 정상 박스 기준 알약 하나의 픽셀 수
pill_px = (ann_df["area"] * ann_df["fill"])[ann_df["fill"] > 0.05]  # 위치 틀린 박스 제외
print("알약 픽셀 수  1%:", int(pill_px.quantile(0.01)),
      "/ 5%:", int(pill_px.quantile(0.05)),
      "/ 중앙값:", int(pill_px.median()))

thr = pill_px.quantile(0.01) * 0.5  # 제일 작은 알약의 절반도 안 되면 잡음으로 봄
print("임계값:", int(thr))

# 2) 성적표: 이미 아는 13장 중 몇 장 잡았나
caught = set(blob_df.loc[blob_df["제일 큰 덩어리(px)"] >= thr, "file_name"])
known = set(issue_imgs)
print()
print(f"이미 아는 문제 {len(known)}장 중 검출: {len(known & caught)}장")
print("놓친 것:", sorted(known - caught))

# 3) 새 후보 (임계값 넘는데 아직 목록에 없는 것)
new_cand = blob_df[~blob_df["이미 찾음"] & (blob_df["제일 큰 덩어리(px)"] >= thr)]
print()
print("새 후보:", len(new_cand))
new_cand.sort_values("제일 큰 덩어리(px)", ascending=False)

검출기 성적: 이미 아는 13장 전부 잡음. 임계값은 제일 작은 알약 크기의 절반.
새 후보 26장은 같은 조합이 각도별로 같이 나옴 → 특정 약의 특징일 가능성.

In [ ]:
from collections import Counter

# 새 후보에 자주 나오는 약 코드
combos = new_cand["file_name"].str.split("_").str[0]
codes = Counter(c for combo in combos for c in combo.split("-")[1:])
print("새 후보 26장에 자주 나오는 약 코드:", codes.most_common(5))

def leftover_mask(name, boxes, margin=10):
    """박스 밖에 남은 큰 덩어리만 True인 마스크 (unboxed_blobs와 같은 기준)"""
    im = np.asarray(Image.open(TRAIN_IMG_DIR / name).convert("RGB")).astype(float)
    bg = np.median(im.reshape(-1, 3), axis=0)
    mask = np.linalg.norm(im - bg, axis=2) > 40
    for _, a in boxes.iterrows():
        x, y = max(0, int(a["x"]) - margin), max(0, int(a["y"]) - margin)
        mask[y:int(a["y2"]) + margin, x:int(a["x2"]) + margin] = False

    labels, _ = ndimage.label(mask)
    sizes = np.bincount(labels.ravel())
    h, w = mask.shape
    keep = []
    for i, (ys, xs) in enumerate(ndimage.find_objects(labels), start=1):
        if sizes[i] >= thr and ys.start > 0 and xs.start > 0 and ys.stop < h and xs.stop < w:
            keep.append(i)
    return np.isin(labels, keep)

# 덩어리 큰 순서로 8장
top_names = new_cand.sort_values("제일 큰 덩어리(px)", ascending=False)["file_name"].head(8).tolist()

fig, axes = plt.subplots(2, 4, figsize=(16, 11))
for ax, name in zip(axes.flat, top_names):
    boxes = ann_df[ann_df["file_name"] == name]
    ax.imshow(Image.open(TRAIN_IMG_DIR / name))
    ax.imshow(np.ma.masked_where(~leftover_mask(name, boxes), np.ones((1280, 976))),
              cmap="autumn", alpha=0.6)  # 박스 밖 덩어리를 빨갛게
    for _, a in boxes.iterrows():
        ax.add_patch(patches.Rectangle((a["x"], a["y"]), a["w"], a["h"],
                                       fill=False, edgecolor="lime", linewidth=2))
    idx = img_df.index[img_df["file_name"] == name][0]
    ax.set_title(f"[{idx}] {name[:30]}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def classify_blobs(name, boxes, margin=10):
    """박스 밖 큰 덩어리를 그림자(배경보다 어두움)와 나머지로 나눠서 셈"""
    im = np.asarray(Image.open(TRAIN_IMG_DIR / name).convert("RGB")).astype(float)
    bg = np.median(im.reshape(-1, 3), axis=0)
    gray, bg_gray = im.mean(axis=2), bg.mean()   # 밝기
    mask = np.linalg.norm(im - bg, axis=2) > 40
    for _, a in boxes.iterrows():
        x, y = max(0, int(a["x"]) - margin), max(0, int(a["y"]) - margin)
        mask[y:int(a["y2"]) + margin, x:int(a["x2"]) + margin] = False

    labels, _ = ndimage.label(mask)
    sizes = np.bincount(labels.ravel())
    h, w = mask.shape
    shadow, other = 0, 0
    for i, sl in enumerate(ndimage.find_objects(labels), start=1):
        ys, xs = sl
        if sizes[i] < thr or ys.start == 0 or xs.start == 0 or ys.stop == h or xs.stop == w:
            continue
        blob_gray = gray[sl][labels[sl] == i].mean()  # 덩어리 평균 밝기 (해당 범위만 봐서 빠름)
        if blob_gray < bg_gray:
            shadow += 1
        else:
            other += 1
    return shadow, other

res = [classify_blobs(n, ann_df[ann_df["file_name"] == n]) for n in new_cand["file_name"]]
new_cand = new_cand.copy()
new_cand["그림자"], new_cand["그림자 아님"] = zip(*res)

print("그림자만 있는 이미지:", (new_cand["그림자 아님"] == 0).sum())
print("그림자 아닌 덩어리 있는 이미지:", (new_cand["그림자 아님"] > 0).sum())
new_cand[new_cand["그림자 아님"] > 0]

In [ ]:
# 정답 아는 13장으로 그림자 판별 규칙 검증
known_res = []
for n in issue_imgs:
    s, o = classify_blobs(n, ann_df[ann_df["file_name"] == n])
    idx = img_df.index[img_df["file_name"] == n][0]
    known_res.append((idx, n, s, o))

known_check = pd.DataFrame(known_res, columns=["idx", "file_name", "그림자", "그림자 아님"])
print("진짜 알약을 '그림자 아님'으로 잡은 이미지:", (known_check["그림자 아님"] > 0).sum(), "/ 13")
known_check

밝기 기준은 실패 (13장 중 3장만 맞힘). 주황색 알약이 밝기 숫자로는 배경보다 낮게 나와서 그림자로 분류됨.
그림자 = 배경과 같은 색인데 어두운 것, 알약 = 배경과 다른 색. 색 비율로 다시 판별한다.
기준값은 정답 아는 두 그룹(진짜 알약 13장, 눈으로 확인한 그림자 8장)으로 정한다.

In [ ]:
def blob_color_diffs(name, boxes, margin=10):
    """박스 밖 큰 덩어리마다 '배경과의 색 차이'를 구함 (밝기는 빼고 색 비율만 비교)"""
    im = np.asarray(Image.open(TRAIN_IMG_DIR / name).convert("RGB")).astype(float)
    bg = np.median(im.reshape(-1, 3), axis=0)
    mask = np.linalg.norm(im - bg, axis=2) > 40
    for _, a in boxes.iterrows():
        x, y = max(0, int(a["x"]) - margin), max(0, int(a["y"]) - margin)
        mask[y:int(a["y2"]) + margin, x:int(a["x2"]) + margin] = False

    chroma = im / (im.sum(axis=2, keepdims=True) + 1e-6)  # 픽셀별 색 비율 (R, G, B 합이 1)
    bg_chroma = bg / bg.sum()

    labels, _ = ndimage.label(mask)
    sizes = np.bincount(labels.ravel())
    h, w = mask.shape
    diffs = []
    for i, sl in enumerate(ndimage.find_objects(labels), start=1):
        ys, xs = sl
        if sizes[i] < thr or ys.start == 0 or xs.start == 0 or ys.stop == h or xs.stop == w:
            continue
        blob_chroma = chroma[sl][labels[sl] == i].mean(axis=0)
        diffs.append(np.linalg.norm(blob_chroma - bg_chroma))
    return diffs

def max_diff(names):
    """이미지마다 가장 색이 다른 덩어리의 색 차이"""
    out = {}
    for n in names:
        d = blob_color_diffs(n, ann_df[ann_df["file_name"] == n])
        out[n] = max(d) if d else np.nan
    return pd.Series(out)

pill_diff = max_diff(issue_imgs)      # 진짜 알약 그룹
shadow_diff = max_diff(top_names)     # 진짜 그림자 그룹

print("진짜 알약  색 차이:", np.sort(pill_diff.values).round(3))
print("진짜 그림자 색 차이:", np.sort(shadow_diff.values).round(3))
print()
print("알약 최솟값:", round(pill_diff.min(), 3), "/ 그림자 최댓값:", round(shadow_diff.max(), 3))

색 차이로 완전히 갈림. 그림자 0.024~0.034, 진짜 알약 0.084~0.26.
기준값 0.06 (두 그룹 사이 빈 구간 가운데).

In [ ]:
COLOR_THR = 0.06

cand_diff = max_diff(new_cand["file_name"])  # 후보 26장, 이미지별 가장 색이 다른 덩어리
new_cand["색 차이"] = new_cand["file_name"].map(cand_diff).round(3)
new_cand["판정"] = np.where(new_cand["색 차이"] > COLOR_THR, "알약 의심", "그림자")

print(new_cand["판정"].value_counts())
print()
print("색 차이 범위:", new_cand["색 차이"].min(), "~", new_cand["색 차이"].max())
new_cand[new_cand["판정"] == "알약 의심"][["file_name", "제일 큰 덩어리(px)", "색 차이"]]

In [ ]:
name = new_cand.loc[new_cand["판정"] == "알약 의심", "file_name"].iloc[0]
boxes = ann_df[ann_df["file_name"] == name]
idx = img_df.index[img_df["file_name"] == name][0]

print(f"[{idx}] {name}")
print(boxes[["drug_name", "x", "y", "w", "h", "fill"]].round(2))

fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(Image.open(TRAIN_IMG_DIR / name))
ax.imshow(np.ma.masked_where(~leftover_mask(name, boxes), np.ones((1280, 976))),
          cmap="autumn", alpha=0.6)  # 박스 밖 덩어리 빨갛게
for _, a in boxes.iterrows():
    ax.add_patch(patches.Rectangle((a["x"], a["y"]), a["w"], a["h"],
                                   fill=False, edgecolor="lime", linewidth=2))
    ax.text(a["x"], a["y"] - 10, a["drug_name"], color="lime", fontsize=9)
ax.set_title(f"[{idx}]", fontsize=10)
ax.axis("off")
plt.show()

[113] 콜리네이트연질캡슐 박스가 캡슐 아래쪽을 못 덮음 (박스 잘림).
fill은 정상이라 앞 검사로는 안 잡혔고, 박스 밖 알약 검사로 찾음.
mAP 0.75~0.95 기준이라 이런 오류는 모델이 맞혀도 점수가 깎임.

한계: 덩어리 기준(약 8,800px)보다 작게 어긋난 박스는 이 방법으로 못 잡음.

In [ ]:
issues.append((img_df.loc[113, "file_name"], "박스 잘림"))

issue_df = pd.DataFrame(issues, columns=["file_name", "문제"])
issue_imgs = issue_df["file_name"].unique()

print("문제 이미지 수:", len(issue_imgs))
print()
print(issue_df["문제"].value_counts())

## 9. 파일 이름 패턴

뒤쪽 숫자 중 70/000/200은 camera_la, camera_lo, size. 앞의 4자리(0_2_0_2)도 촬영 조건일 것.
값 종류를 세고, test 이름도 같은 형식인지 본다.

In [ ]:
# 파일 이름을 _ 기준으로 쪼개서 자리별 값 세기 (0번 자리는 약 코드라 제외)
parts = img_df["file_name"].str.replace(".png", "").str.split("_", expand=True)
print("train 이름 자리 수:", parts.shape[1])
for c in parts.columns[1:]:
    print(f"  자리 {c}:", parts[c].value_counts().to_dict())

print()
print("test 이름 예시:")
for p in test_imgs[:5]:
    print(" ", p.name)

train 이름 8자리 중 바뀌는 건 각도(70/75/90)와 3번 자리(0이 228장, 1이 4장)뿐.
촬영 조건이 거의 하나로 통일됨 → test 조건이 다르면 문제될 수 있음.
test 이름은 번호뿐이라 이미지 자체로 비교해야 함.

In [ ]:
# json에서 촬영 조건 정보 다시 모으기 (이미지당 하나면 되니까 이미지 기준으로 중복 제거)
meta = []
for jp in json_files:
    with open(jp, encoding="utf-8") as f:
        img = json.load(f)["images"][0]
    meta.append({k: img[k] for k in ["file_name", "drug_S", "drug_dir", "light_color", "back_color"]})
meta_df = pd.DataFrame(meta)

meta_df["자리3"] = meta_df["file_name"].str.split("_").str[3]

# 자리3 값별로 각 조건이 어떻게 나뉘는지 (json 기준이라 알약 하나당 한 줄)
for col in ["drug_S", "drug_dir", "light_color", "back_color"]:
    print(pd.crosstab(meta_df["자리3"], meta_df[col]))
    print()

3번 자리 = 앞면(0)/뒷면(1). 뒷면은 4장뿐.
조명, 배경, 알약 상태는 전부 한 가지. train 촬영 조건이 아주 단조로움.
뒷면은 augmentation으로 못 만듦 → test에 뒷면이 많으면 외부 데이터 후보.

## 10. train vs test 비교

In [ ]:
def img_stats(p):
    with Image.open(p) as im:
        w, h = im.size
        small = np.asarray(im.convert("RGB").reduce(4)).astype(float)  # 4배 줄여서 빠르게
    bg = np.median(small.reshape(-1, 3), axis=0)  # 배경색 (알약이 작아서 중앙값 ≈ 배경)
    return w, h, *bg

cols = ["w", "h", "bg_R", "bg_G", "bg_B"]
train_stats = pd.DataFrame([img_stats(p) for p in train_imgs], columns=cols)
test_stats = pd.DataFrame([img_stats(p) for p in test_imgs], columns=cols)

print("test 해상도:")
print(test_stats.groupby(["w", "h"]).size())
print()

# 배경색 비교 (R, G, B 각각의 분포)
compare = pd.concat({
    "train": train_stats[["bg_R", "bg_G", "bg_B"]].describe().loc[["min", "50%", "max"]],
    "test": test_stats[["bg_R", "bg_G", "bg_B"]].describe().loc[["min", "50%", "max"]],
})
compare.round(1)

test 해상도도 전부 976×1280.
배경색은 비슷한데 test가 살짝 어둡고, train에 없던 더 어두운 값도 있음.
몇 장이나 되는지, 어떻게 생겼는지 본다.

In [ ]:
# train 배경색 범위를 벗어나는 test 사진
lo = train_stats[["bg_R", "bg_G", "bg_B"]].min()
hi = train_stats[["bg_R", "bg_G", "bg_B"]].max()
outside = ((test_stats[["bg_R", "bg_G", "bg_B"]] < lo) | (test_stats[["bg_R", "bg_G", "bg_B"]] > hi)).any(axis=1)
print(f"train 배경색 범위 밖 test 사진: {outside.sum()}장 / {len(test_stats)}장")

# 배경 밝기(RGB 평균) 기준으로 비교
train_stats["bright"] = train_stats[["bg_R", "bg_G", "bg_B"]].mean(axis=1)
test_stats["bright"] = test_stats[["bg_R", "bg_G", "bg_B"]].mean(axis=1)

# 가장 어두운 test 4장 vs 가장 어두운 train 4장
dark_test = test_stats.nsmallest(4, "bright").index
dark_train = train_stats.nsmallest(4, "bright").index

fig, axes = plt.subplots(2, 4, figsize=(16, 11))
for ax, i in zip(axes[0], dark_train):
    ax.imshow(Image.open(train_imgs[i]))
    ax.set_title(f"train 어두운 쪽 / 밝기 {train_stats.loc[i, 'bright']:.0f}", fontsize=10)
    ax.axis("off")
for ax, i in zip(axes[1], dark_test):
    ax.imshow(Image.open(test_imgs[i]))
    ax.set_title(f"test {test_imgs[i].name} / 밝기 {test_stats.loc[i, 'bright']:.0f}", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

test 56장(6.7%)이 train 배경색 범위 밖. 근데 그림으로 보면 같은 세팅에 조명만 살짝 어두운 정도.
밝기/대비 augmentation으로 충분히 대응 가능해 보임.
test에 "Hanmi", "K 50" 같은 알약이 보임. train에 없는 클래스일 수 있음.

## 11. 클래스 분포

In [ ]:
# 약 종류별 박스 수, 등장 이미지 수
cls = ann_df.groupby(["category_id", "drug_name"]).agg(
    박스수=("file_name", "size"),
    이미지수=("file_name", "nunique"),
).reset_index().sort_values("박스수", ascending=False)

print("약 종류 수:", len(cls))
print()
print("박스 수 분포:")
print(cls["박스수"].describe().round(1))
print()
for n in [3, 5, 10]:
    print(f"박스 {n}개 이하인 약: {(cls['박스수'] <= n).sum()}종")
print()

# 상위 5개, 하위 5개
print(cls.head(5).to_string(index=False))
print("...")
print(cls.tail(5).to_string(index=False))

약 56종. 일양하이트린정 153개 vs 중앙값 9개로 불균형 심함.
박스 3개뿐인 약 17종 = 조합 하나를 3각도로 찍은 게 전부.
조합 단위로 나누면 이런 약은 train이나 val 한쪽에만 있게 됨. 분할 전에 확인 필요.

In [ ]:
# 약별로 몇 개의 서로 다른 조합에 나오는지
ann_df["combo"] = ann_df["file_name"].str.split("_").str[0]
combo_per_cls = ann_df.groupby("category_id")["combo"].nunique()

print("전체 조합 수:", ann_df["combo"].nunique())
print()
print("약별 조합 수 분포 (조합 수: 약 종류 수):")
print(combo_per_cls.value_counts().sort_index().to_dict())
print()
print("조합 1개뿐인 약:", (combo_per_cls == 1).sum(), "종")

# 조합별 사진 수 (각도 3개가 다 있는지)
imgs_per_combo = img_df.groupby("combo").size()
print()
print("조합별 사진 수 분포:", imgs_per_combo.value_counts().sort_index().to_dict())

실제 독립 샘플은 114개 조합. 각도만 다른 사진은 사실상 같은 정보.
조합 1개뿐인 약 17종 → 이 약이 들어간 조합은 무조건 train.
남은 조합에서 val을 뽑을 수 있는지 확인.

## 12. train/val 분할 준비

In [ ]:
# 조합 1개뿐인 약 → 그 약이 들어간 조합은 train 고정
rare_cls = combo_per_cls[combo_per_cls == 1].index
must_train = set(ann_df.loc[ann_df["category_id"].isin(rare_cls), "combo"])

all_combos = set(img_df["combo"])
free = all_combos - must_train  # val 후보가 될 수 있는 조합

print(f"train 고정 조합: {len(must_train)}개 / 사진 {img_df['combo'].isin(must_train).sum()}장")
print(f"val 후보 조합: {len(free)}개 / 사진 {img_df['combo'].isin(free).sum()}장")

# 참고: 라벨 문제 있는 14장이 어느 쪽에 속하나
issue_combos = set(img_df.loc[img_df["file_name"].isin(issue_imgs), "combo"])
print()
print("문제 이미지가 있는 조합:", len(issue_combos), "개",
      "/ 그중 val 후보:", len(issue_combos & free), "개")

조합 1개뿐인 약 17종은 13개 조합(39장)에 몰려 있음 → train 고정.
라벨 문제 조합 14개 중 12개가 val 후보. val 라벨이 틀리면 점수 자체가 틀어지니, 고치기 전엔 val에서 제외.

분할 규칙
1. 조합 단위로 나눔
2. 조합 1개뿐인 약이 든 조합은 train
3. 라벨 문제 조합은 val 제외
4. val에 넣는 약은 train에도 최소 한 조합 남김
5. val은 사진 기준 약 20%

In [ ]:
rng = np.random.default_rng(42)  # 시드 고정 → 누가 돌려도 같은 분할

combo_cls = ann_df.groupby("combo")["category_id"].apply(set)  # 조합별 들어 있는 약
left_in_train = combo_per_cls.copy()                            # 약별 train에 남은 조합 수

cand = sorted(free - issue_combos)
rng.shuffle(cand)

target = round(len(img_df) * 0.2)
val_combos, val_n = [], 0
for c in cand:
    if val_n >= target:
        break
    # 이 조합을 val로 보내도 모든 약이 train에 한 조합 이상 남을 때만
    if all(left_in_train[k] > 1 for k in combo_cls[c]):
        val_combos.append(c)
        val_n += imgs_per_combo[c]
        for k in combo_cls[c]:
            left_in_train[k] -= 1

img_df["split"] = np.where(img_df["combo"].isin(val_combos), "val", "train")
ann_df["split"] = ann_df["combo"].map(img_df.drop_duplicates("combo").set_index("combo")["split"])

print("val 조합:", len(val_combos), "/ val 사진:", (img_df["split"] == "val").sum(),
      "/ train 사진:", (img_df["split"] == "train").sum())
print()
print("train에 있는 약:", ann_df.loc[ann_df["split"] == "train", "category_id"].nunique(), "/ 56")
print("val에 있는 약:", ann_df.loc[ann_df["split"] == "val", "category_id"].nunique(), "/ 56")
print("val에만 있는 약:", set(ann_df.loc[ann_df["split"] == "val", "category_id"])
      - set(ann_df.loc[ann_df["split"] == "train", "category_id"]))

val 46장(19.8%) / train 186장. train에 56종 전부, val에만 있는 약 없음.

한계
- val엔 23종만 있음. val mAP는 이 23종 기준.
- val이 22개 조합뿐이라 점수가 흔들림. 1~2점 차이는 우연일 수 있으니 중요한 비교는 시드 바꿔서 여러 번.
- test에만 있는 약은 val로 측정 불가.

In [ ]:
# 일양하이트린정 비율 비교
for s in ["train", "val"]:
    part = ann_df[ann_df["split"] == s]
    share = (part["category_id"] == 3351).mean()
    print(f"{s}: 박스 {len(part)}개 / 일양하이트린정 비율 {share:.1%}")
print()

# 각도 분포 비교
print(pd.crosstab(ann_df["split"], ann_df["camera_la"], normalize="index").round(2))
print()

# val에서 박스 많은 약 상위 5개
print(ann_df[ann_df["split"] == "val"]["drug_name"].value_counts().head(5))

각도 분포는 train/val 비슷함.
일양하이트린정 비율이 train 18.3% vs val 27.8%로 쏠림. 흔한 약이 든 조합이 규칙을 통과하기 쉬워서 생긴 것.
시드 500개로 나눠보고, 일양 비율이 전체 평균에 가깝고 val 약 종류가 많은 분할을 고른다.

In [ ]:
# 조합 단위 요약 (분할 계산을 빠르게)
combo_info = ann_df.groupby("combo").agg(
    n_box=("category_id", "size"),
    n_3351=("category_id", lambda s: (s == 3351).sum()),
)
combo_info["n_img"] = imgs_per_combo
combo_info["cls"] = combo_cls
overall_share = (ann_df["category_id"] == 3351).mean()

def make_split(seed):
    """규칙 1~5대로 val 조합 목록을 만든다."""
    rng = np.random.default_rng(seed)
    left = combo_per_cls.copy()
    cand = sorted(free - issue_combos)
    rng.shuffle(cand)
    val, n = [], 0
    for c in cand:
        if n >= target:
            break
        if all(left[k] > 1 for k in combo_info.at[c, "cls"]):
            val.append(c)
            n += combo_info.at[c, "n_img"]
            for k in combo_info.at[c, "cls"]:
                left[k] -= 1
    return val

results = []
for seed in range(500):
    v = combo_info.loc[make_split(seed)]
    share = v["n_3351"].sum() / v["n_box"].sum()
    n_cls = len(set().union(*v["cls"]))
    results.append((seed, v["n_img"].sum(), round(share, 3), n_cls))

res_df = pd.DataFrame(results, columns=["seed", "val 사진", "일양 비율", "val 약 종류"])
res_df["비율 차이"] = (res_df["일양 비율"] - overall_share).abs()

print(f"전체 일양 비율: {overall_share:.3f}")
print(f"val 약 종류 범위: {res_df['val 약 종류'].min()} ~ {res_df['val 약 종류'].max()}")
print()
# 비율 차이 작은 순, 같으면 약 종류 많은 순
res_df.sort_values(["비율 차이", "val 약 종류"], ascending=[True, False]).head(10)

시드 바꾸니 일양 비율 0.203(전체 0.201)까지 맞춰짐. val 약 종류도 23 → 28.
최종 기준: 일양 비율 ±0.02 안에서 val 약 종류가 가장 많은 시드.

In [ ]:
# 비율 ±0.02 안에서 약 종류 많은 순 → 비율 차이 작은 순
ok = res_df[res_df["비율 차이"] <= 0.02].sort_values(["val 약 종류", "비율 차이"], ascending=[False, True])
print(ok.head(5).to_string(index=False))

BEST_SEED = int(ok.iloc[0]["seed"])
print()
print("선택한 시드:", BEST_SEED)

# 최종 분할 적용
final_val = make_split(BEST_SEED)
img_df["split"] = np.where(img_df["combo"].isin(final_val), "val", "train")
ann_df["split"] = ann_df["combo"].map(img_df.drop_duplicates("combo").set_index("combo")["split"])

# 최종 확인
tr, va = ann_df[ann_df["split"] == "train"], ann_df[ann_df["split"] == "val"]
print(f"train 사진 {(img_df['split'] == 'train').sum()} / val 사진 {(img_df['split'] == 'val').sum()}")
print(f"train 약 {tr['category_id'].nunique()}종 / val 약 {va['category_id'].nunique()}종")
print("val에만 있는 약:", set(va["category_id"]) - set(tr["category_id"]))
print(f"일양 비율 train {(tr['category_id'] == 3351).mean():.3f} / val {(va['category_id'] == 3351).mean():.3f}")
print("val에 문제 이미지:", img_df.loc[img_df["split"] == "val", "file_name"].isin(issue_imgs).sum())

# 저장 (data/processed는 git에 안 올라감)
out_path = ROOT / "data" / "processed" / "split.csv"
img_df[["file_name", "combo", "split"]].to_csv(out_path, index=False)
print()
print("저장:", out_path)

최종 분할: 시드 410. train 185 / val 47장, val 약 30종, val에 문제 이미지 없음.
data/processed/split.csv에 저장 (git엔 안 올라가니 시드와 규칙으로 재현).

## 13. 학습 이미지 크기(imgsz)

In [ ]:
# 박스 짧은 변 길이 (알약이 작을수록 검출 어려움)
short_side = ann_df[["w", "h"]].min(axis=1)

# YOLO는 긴 변(1280)을 imgsz에 맞춰 줄임
rows = []
for imgsz in [640, 960, 1280]:
    scale = imgsz / 1280
    s = short_side * scale
    rows.append({
        "imgsz": imgsz,
        "최소(px)": round(s.min()),
        "5%(px)": round(s.quantile(0.05)),
        "중앙값(px)": round(s.median()),
        "32px 미만 박스": int((s < 32).sum()),  # COCO 기준 '작은 물체'
    })

print("원본 박스 짧은 변: 최소", round(short_side.min()), "/ 중앙값", round(short_side.median()))
print()
pd.DataFrame(rows)

## 14. 정리

### 라벨 품질
- 문제 이미지 14장 / 232장. 유형: 라벨 누락 8, 다른 약에 같은 박스 복사 3, 박스 이미지 밖 1, 박스가 빈 곳에 있음 1, 박스가 알약을 다 못 덮음 1
- 13장 중 11장에 일양하이트린정(K-003351)이 들어 있음. 이 약 쪽 라벨 작업에서 실수가 몰림
- 평가가 mAP 0.75~0.95라 박스 오류가 치명적. 버리기보다 박스 14개 정도 다시 그리는 걸 추천
- 고치기 전까지는 val에 넣지 않음 (val 라벨이 틀리면 점수 자체가 틀어짐)
- 한계: 몇십 px 정도 어긋난 박스는 자동 검사로 못 잡음. 다른 팀원 결과(19장)와 목록 대조 필요

### 클래스
- 56종. 일양하이트린정 153개 vs 중앙값 9개로 불균형 심함
- 박스 3개뿐인 약 17종 = 알약 한 번 놓고 3각도로 찍은 게 전부
- 실제 독립 샘플은 232장이 아니라 114개 조합

### train/val 분할
- 조합 단위로 나눔 (각도만 다른 사진이 양쪽에 들어가면 val 점수가 부풀음)
- 조합 1개뿐인 약이 든 조합은 train 고정, 문제 이미지 조합은 val 제외, val에 넣는 약은 train에도 남김
- 시드 500개 중 일양 비율이 평균에 가깝고 val 약 종류가 많은 시드 410 선택
- train 185 / val 47장, val 약 30종
- 한계: val이 작아서 점수가 흔들림. 1~2점 차이는 우연일 수 있으니 중요한 비교는 여러 시드로

### 촬영 조건, train vs test
- train은 배경·조명·알약 상태가 전부 한 가지. 각도 70/75/90만 바뀜
- 뒷면 사진은 4장뿐. 뒷면은 augmentation으로 못 만듦
- test도 전부 976×1280. 56장(6.7%)이 train보다 살짝 어두움 → 밝기/대비 augmentation으로 대응
- 두꺼운 알약은 그림자가 크게 짐. 모델이 그림자까지 박스에 넣는지 나중에 확인

### 외부 데이터
- 대회 측이 test에 train에 없는 약이 있다고 명시. 희귀한 약 17종, 뒷면 부족까지 생각하면 외부 데이터 필요성 높음
- AI Hub 데이터 사용 가능. 단 TL_2_조합.zip, TS_2_조합.zip은 금지. 데이터셋마다 클래스 구성이 다르니 확인 필요

### imgsz
- 640으로 줄여도 박스 최소 62px라 검출 자체는 문제없음
- 다만 약 구분은 새겨진 글자에 달려 있고 평가도 박스 정밀도를 요구함 → 640으로 기준선 잡고 960, 1280 실험 추천